# FINS notebook 11b — a tracer in a hidden flow, the learned arm completed (exp69; Colab, GPU)

Notebook 11's emulators are accurate (one-step error 0.35-0.48 %), but their float32 rollouts on the GPU diverged
(|c| up to 1e5-1e10 within the 47 steps), so the self arm audited non-physical states; and seed 42 has no FINS result.
This notebook:
* cell 2: makes sure the three emulators exist on Drive (trains the missing ones with notebook 11's recipe; seed 42's
  checkpoint may have been lost when Drive was full). Check that Drive has at least 1 GB free before running; files that Colab deletes or overwrites on Drive go to the Trash and still count, so empty the Trash first. The trainer writes its .partial every 5000 steps here (about 0.3 GB each).
* cell 3: per emulator, the rollout check (float32 and float64, without and with pyqg's filter after each step: distance
  to the data, max |c|), then the self arm in float64 (`self64`, development's precision), the self arm in float64 with
  the filter (`selffilt`, a post hoc control) and the data arm where it is missing. A few minutes per emulator.
* cell 4 shows progress; cell 5 zips the json and logs to `fins_nb11b.zip` (upload this one to the chat).

Before running, upload the newest **`fins_repo_full.zip`** from the chat to My Drive (root or one folder down).

In [ ]:
# 1. setup: Drive, the newest repository zip (with exp69's rollcheck), pyqg from GitHub, GPU check, Drive space
import os, sys, subprocess, zipfile, glob, time, shutil
from google.colab import drive
drive.mount('/content/drive')
os.environ['FINS_RESULTS_DIR'] = '/content/drive/MyDrive/fins_runs/results'
os.environ['TQDM_DISABLE'] = '1'
R = os.environ['FINS_RESULTS_DIR']
os.makedirs(os.path.join(R, 'tracer'), exist_ok=True)


def newest(pattern, test):
    cands = sorted(glob.glob(f'/content/drive/MyDrive/{pattern}') + glob.glob(f'/content/drive/MyDrive/*/{pattern}'),
                   key=os.path.getmtime, reverse=True)
    for z in cands:
        ok = test(z)
        print(f'{z}  {time.ctime(os.path.getmtime(z))}  {os.path.getsize(z)} bytes  usable: {ok}')
        if ok:
            return z


def has_rollcheck(z):
    with zipfile.ZipFile(z) as zz:
        return b'def rollcheck' in zz.read('fins_repo/experiments/exp69_hidden_tracer.py')


ZIP_PATH = newest('fins_repo_full*.zip', has_rollcheck)
if ZIP_PATH is None:
    raise SystemExit('no fins_repo_full*.zip on Drive has exp69 rollcheck: upload the newest zip from the chat')
print('using', ZIP_PATH)
shutil.rmtree('/content/fins', ignore_errors=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall('/content/fins')
EXPDIR = '/content/fins/fins_repo/experiments'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'git+https://github.com/pyqg/pyqg.git'], check=False)
chk = subprocess.run([sys.executable, '-c', 'import exp69_hidden_tracer'], cwd=EXPDIR, capture_output=True, text=True)
if chk.returncode:
    raise SystemExit('exp69 does not import here:\n' + chk.stderr[-2000:])
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
if not torch.cuda.is_available():
    print('NO GPU: Runtime -> Change runtime type -> T4 GPU (or better), then rerun this cell')
total, used, free = shutil.disk_usage('/content/drive/MyDrive')
print(f'Drive: {free / 2 ** 30:.2f} GB free of {total / 2 ** 30:.1f} GB (a checkpoint is about 0.1 GB, 0.3 GB while training)')
CACHE = '/content/tracer_cache'
RUNS = os.path.join(R, 'tracer_runs')
os.makedirs(os.path.join(RUNS, 'logs'), exist_ok=True)
SEEDS = (42, 43, 44)
COMMON = ['--n', '128', '--w', '4', '--cache', CACHE]


def go(stem, cmd, keys):
    if os.path.exists(stem + '.json'):
        print('done', os.path.basename(stem))
        return
    log = os.path.join(RUNS, 'logs', os.path.basename(stem) + '.log')
    t0 = time.time()
    with open(log, 'a') as fh:
        fh.write('$ ' + ' '.join(cmd) + '\n')
        fh.flush()
        rc = subprocess.call([sys.executable, '-u', *cmd], cwd=EXPDIR, stdout=fh, stderr=subprocess.STDOUT)
    lines = open(log).read().splitlines()
    keep = [l.strip() for l in lines if any(k in l for k in keys)] if rc == 0 else lines[-8:]
    print(os.path.basename(stem), 'exit', rc, f'{time.time() - t0:.0f} s', *[k[:170] for k in keep[-10:]], sep='\n   ')


def ckpt(s):
    return os.path.join(R, 'tracer', f'wfno_n128_k0.05_w4_s{s}.pt')


for s in SEEDS:
    print(f's{s}: checkpoint', 'present' if os.path.exists(ckpt(s)) else 'MISSING (cell 2 trains it)')

In [ ]:
# 2. train any missing emulator (notebook 11's recipe; resumable)
for s in SEEDS:
    if os.path.exists(ckpt(s)):
        print('trained', os.path.basename(ckpt(s)))
        continue
    log = os.path.join(RUNS, 'logs', f'train_w4_s{s}.log')
    t0 = time.time()
    print(f's{s}: training (log {log})', flush=True)
    with open(log, 'a') as fh:
        cmd = ['exp69_hidden_tracer.py', 'train', '--seed', str(s), *COMMON, '--modes', '40', '--n-traj', '1024',
               '--steps', '10000', '--batch', '32', '--log-every', '500', '--ckpt-every', '5000']
        fh.write('$ ' + ' '.join(cmd) + '\n')
        fh.flush()
        p = subprocess.Popen([sys.executable, '-u', *cmd], cwd=EXPDIR, stdout=fh, stderr=subprocess.STDOUT)
        seen = len(open(log).read().splitlines())
        while p.poll() is None:
            time.sleep(60)
            lines = open(log).read().splitlines()
            for l in lines[seen:]:
                print('   ', l.strip()[:170], flush=True)
            seen = len(lines)
    lines = open(log).read().splitlines()
    print(f's{s}: exit {p.returncode}, {time.time() - t0:.0f} s', *[l[:170] for l in lines[seen:]], sep='\n   ')
left = [f's{s}' for s in SEEDS if not os.path.exists(ckpt(s))]
print('all three emulators present' if not left else 'NOT YET: ' + ', '.join(left))

In [ ]:
# 3. per emulator: rollout check, self arm in float64, self arm in float64 with the filter, data arm if missing
keys = ('held-out error', 'own rollouts', 'lowest', 'observed:', 'heat-false', 'H (10', 'span6')
for s in SEEDS:
    if not os.path.exists(ckpt(s)):
        print('not trained yet: s', s)
        continue
    stem = os.path.join(RUNS, f'ex69_rollcheck_s{s}')
    go(stem, ['exp69_hidden_tracer.py', 'rollcheck', '--seed', str(s), *COMMON, '--save', stem], ('rollcheck',))
    for arm, extra in (('self64', ['--arm', 'self', '--fp64']), ('selffilt', ['--arm', 'self', '--fp64', '--roll-filter']),
                       ('data', ['--arm', 'data'])):
        stem = os.path.join(RUNS, f'ex69_{arm}_fno_w4_s{s}')
        go(stem, ['exp69_hidden_tracer.py', 'fins', *extra, '--seed', str(s), *COMMON, '--save', stem], keys)

In [ ]:
# 4. progress
for s in SEEDS:
    have = [a for a in ('rollcheck', 'self64_fno_w4', 'selffilt_fno_w4', 'data_fno_w4', 'self_fno_w4')
            if os.path.exists(os.path.join(RUNS, f'ex69_{a}_s{s}.json'))]
    print(f's{s}:', ', '.join(have) or 'nothing yet')

In [ ]:
# 5. copy the json and logs file by file to local disk (remount if Drive drops); fins_nb11b.zip in /content and on Drive
local = '/content/fins_nb11b'
shutil.rmtree(local, ignore_errors=True)
files = glob.glob(os.path.join(RUNS, '*.json')) + glob.glob(os.path.join(RUNS, 'logs', '*.log'))
for f in sorted(set(files)):
    dst = os.path.join(local, os.path.relpath(f, R))
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    for attempt in range(3):
        try:
            shutil.copy2(f, dst)
            break
        except OSError as e:
            print('retry', f, e)
            drive.mount('/content/drive', force_remount=True)
            time.sleep(5)
out = '/content/fins_nb11b.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(glob.glob(os.path.join(local, '**', '*'), recursive=True)):
        if os.path.isfile(f):
            z.write(f, os.path.relpath(f, local))
print(out, os.path.getsize(out), 'bytes')
try:
    shutil.copy2(out, '/content/drive/MyDrive/fins_nb11b.zip')
    print('copied to MyDrive: fins_nb11b.zip (upload this one to the chat)')
except OSError as e:
    print('Drive copy failed; download /content/fins_nb11b.zip from the file pane:', e)